# System Benchmark v2 — Modern Data Lakehouse

Notebook này đo **4 metrics chính** sau khi toàn bộ pipeline đã chạy xong:

| # | Metric | Phương pháp |
|---|---|---|
| 1 | **End-to-end Ingestion Latency** | Đọc từ Bronze snapshot history — khoảng thời gian giữa 2 batch commit liên tiếp |
| 2 | **SCD2 MERGE Throughput** | Đọc từ Silver snapshot history — added/deleted records per second |
| 3 | **Compaction Impact** | Số file + query time trước/sau REWRITE DATA FILES |
| 4 | **Gold Query Latency** | Thời gian analytics query trên Gold layer |

> **v2 Fix:** Metric 1 & 2 không dùng probe/polling hay MERGE trực tiếp nữa.
> Thay vào đó đọc từ Iceberg snapshot metadata — hoạt động ngay cả khi benchmark
> đang reuse SparkSession của Bronze/Silver (cùng JVM).

Kết quả xuất ra bảng tổng hợp cuối notebook — copy thẳng vào chương **"Đánh giá hệ thống"** của luận văn.

In [1]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("Lakehouse_Benchmark") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.io.ResolvingFileIO") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.defaultCatalog", "nessie") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

hc = spark.sparkContext._jsc.hadoopConfiguration()
hc.set("fs.s3a.endpoint",               "http://minio:9000")
hc.set("fs.s3a.access.key",             "admin")
hc.set("fs.s3a.secret.key",             "password")
hc.set("fs.s3a.path.style.access",      "true")
hc.set("fs.s3a.connection.ssl.enabled", "false")
hc.set("fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark.sparkContext.setLogLevel("ERROR")

# Kết quả benchmark lưu vào dict này
results = {}

print("✅ SparkSession ready — bắt đầu benchmark v2")

✅ SparkSession ready — bắt đầu benchmark v2


26/06/16 02:38:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## Metric 1 — End-to-end Ingestion Latency (Kafka → Bronze)

**Phương pháp v2:** Đọc từ Bronze snapshot history.
Mỗi lần Bronze stream trigger (15s) thì Iceberg commit 1 snapshot mới.
Khoảng thời gian giữa 2 snapshot liên tiếp = batch interval thực tế của pipeline.

Không cần stream đang chạy, không block session — đọc hoàn toàn từ metadata.

In [2]:
print("=" * 55)
print("METRIC 1: END-TO-END INGESTION LATENCY")
print("=" * 55)

# Lấy toàn bộ snapshot history của Bronze, tính interval giữa các batch
bronze_snapshots = spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        CAST(summary['added-records']   AS LONG) AS added_records,
        CAST(summary['total-records']   AS LONG) AS total_records,
        summary['spark.app.id']                  AS app_id
    FROM nessie.bronze.yelp_users.snapshots
    ORDER BY committed_at ASC
""").collect()

print(f"  Tổng số snapshots Bronze : {len(bronze_snapshots)}")
print()

# Tính batch interval giữa các snapshot liên tiếp
intervals = []
for i in range(1, len(bronze_snapshots)):
    prev = bronze_snapshots[i - 1]
    curr = bronze_snapshots[i]
    delta_sec = (curr["committed_at"] - prev["committed_at"]).total_seconds()
    intervals.append(delta_sec)

if intervals:
    avg_interval = sum(intervals) / len(intervals)
    min_interval = min(intervals)
    max_interval = max(intervals)
    # Loại bỏ outlier (lần đầu chạy thường chậm hơn do cold start)
    stable_intervals = sorted(intervals)[1:-1] if len(intervals) > 4 else intervals
    median_interval  = sorted(stable_intervals)[len(stable_intervals) // 2]

    print(f"  Batch interval (avg)     : {avg_interval:.1f}s")
    print(f"  Batch interval (median)  : {median_interval:.1f}s  ← dùng giá trị này cho luận văn")
    print(f"  Batch interval (min)     : {min_interval:.1f}s")
    print(f"  Batch interval (max)     : {max_interval:.1f}s")
    print()
    print(f"  Trigger config           : 15s")
    print(f"  → End-to-end latency ≈ median batch interval = {median_interval:.1f}s")
    print(f"    (record vào Kafka → commit vào Iceberg Bronze)")

    # Hiển thị 5 snapshot cuối để confirm
    print()
    print("  5 snapshots gần nhất:")
    for s in bronze_snapshots[-5:]:
        print(f"    {s['committed_at'].strftime('%H:%M:%S')} | "
              f"+{s['added_records']:>7,} records | "
              f"total: {s['total_records']:>10,}")

    results["ingestion_latency_sec"]      = round(median_interval, 1)
    results["ingestion_latency_avg_sec"]  = round(avg_interval, 1)
    results["ingestion_latency_min_sec"]  = round(min_interval, 1)
    results["ingestion_latency_max_sec"]  = round(max_interval, 1)
    results["bronze_total_snapshots"]     = len(bronze_snapshots)
else:
    print("  ⚠️  Chỉ có 1 snapshot — cần ít nhất 2 snapshot để tính interval")
    results["ingestion_latency_sec"] = None

METRIC 1: END-TO-END INGESTION LATENCY
  Tổng số snapshots Bronze : 174

  Batch interval (avg)     : 21.2s
  Batch interval (median)  : 15.0s  ← dùng giá trị này cho luận văn
  Batch interval (min)     : 13.5s
  Batch interval (max)     : 749.5s

  Trigger config           : 15s
  → End-to-end latency ≈ median batch interval = 15.0s
    (record vào Kafka → commit vào Iceberg Bronze)

  5 snapshots gần nhất:
    02:11:00 | +  6,062 records | total:  2,771,946
    02:11:15 | +  5,439 records | total:  2,777,385
    02:11:30 | +  5,178 records | total:  2,782,563
    02:11:46 | +    492 records | total:  2,783,055
    02:16:31 | +      1 records | total:  2,783,056


---
## Metric 2 — SCD2 MERGE Throughput (Silver Layer)

**Phương pháp v2:** Đọc từ Silver snapshot history.
Mỗi micro-batch của Silver tạo ra 2 snapshot (MERGE step 1 + step 2).
Throughput = tổng records được xử lý / tổng thời gian pipeline chạy.

In [3]:
print("=" * 55)
print("METRIC 2: SCD2 MERGE THROUGHPUT")
print("=" * 55)

# ── 2A: Đọc Silver table stats ──────────────────────────────
silver_stats = spark.sql("""
    SELECT
        COUNT(*)                                              AS total_records,
        COUNT(DISTINCT user_id)                               AS unique_users,
        SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)  AS expired_records,
        SUM(CASE WHEN is_current = true  THEN 1 ELSE 0 END)  AS active_records,
        MIN(effective_time)                                   AS first_record_time,
        MAX(effective_time)                                   AS last_record_time
    FROM nessie.silver.yelp_users_scd2
""").collect()[0]

total_records   = silver_stats["total_records"]
unique_users    = silver_stats["unique_users"]
expired_records = silver_stats["expired_records"]
active_records  = silver_stats["active_records"]
t_first         = silver_stats["first_record_time"]
t_last          = silver_stats["last_record_time"]

print(f"  Total records in Silver  : {total_records:,}")
print(f"  Unique users (SCD keys)  : {unique_users:,}")
print(f"  Active records (current) : {active_records:,}")
print(f"  Expired (SCD2 history)   : {expired_records:,} "
      f"({expired_records/total_records*100:.1f}% of total)")
print()

# ── 2B: Throughput từ Silver snapshot history ────────────────
silver_snap = spark.sql("""
    SELECT
        MIN(committed_at)                                      AS first_commit,
        MAX(committed_at)                                      AS last_commit,
        COUNT(*)                                               AS total_snapshots,
        SUM(CAST(summary['added-records']   AS LONG))         AS total_added,
        SUM(CAST(summary['deleted-records'] AS LONG))         AS total_deleted,
        SUM(COALESCE(CAST(summary['added-records'] AS LONG), 0)
          + COALESCE(CAST(summary['deleted-records'] AS LONG), 0)) AS total_ops
    FROM nessie.silver.yelp_users_scd2.snapshots
""").collect()[0]

first_commit     = silver_snap["first_commit"]
last_commit      = silver_snap["last_commit"]
total_snapshots  = silver_snap["total_snapshots"]
total_added      = silver_snap["total_added"]   or 0
total_deleted    = silver_snap["total_deleted"] or 0
total_ops        = silver_snap["total_ops"]     or 0

if first_commit and last_commit:
    duration_sec = (last_commit - first_commit).total_seconds()
    throughput_rps = total_records / duration_sec if duration_sec > 0 else 0
    ops_rps        = total_ops     / duration_sec if duration_sec > 0 else 0
else:
    duration_sec   = None
    throughput_rps = None
    ops_rps        = None

print(f"  Silver snapshot history:")
print(f"    Total snapshots          : {total_snapshots}  (2 per micro-batch = MERGE step1 + step2)")
print(f"    Total added ops          : {total_added:,}")
print(f"    Total deleted ops        : {total_deleted:,}")
if duration_sec:
    print(f"    Pipeline duration        : {duration_sec:.0f}s ({duration_sec/60:.1f} phút)")
    print(f"    Throughput (records/s)   : {throughput_rps:,.1f}  ← dùng giá trị này cho luận văn")
    print(f"    MERGE ops/s (add+del)    : {ops_rps:,.1f}")
else:
    print("    Pipeline duration        : N/A")

# ── 2C: MERGE latency từ snapshot interval ───────────────────
print()
print("  MERGE latency per micro-batch (từ Silver snapshot intervals):")
silver_snaps_list = spark.sql("""
    SELECT committed_at,
           CAST(summary['added-records']   AS LONG) AS added,
           CAST(summary['deleted-records'] AS LONG) AS deleted
    FROM nessie.silver.yelp_users_scd2.snapshots
    ORDER BY committed_at ASC
""").collect()

# Silver tạo 2 snapshot per batch: pair chúng lại
merge_latencies = []
for i in range(0, len(silver_snaps_list) - 1, 2):
    t_start = silver_snaps_list[i]["committed_at"]
    t_end   = silver_snaps_list[i + 1]["committed_at"]
    delta_ms = (t_end - t_start).total_seconds() * 1000
    merge_latencies.append(delta_ms)

if merge_latencies:
    merge_latencies_sorted = sorted(merge_latencies)
    median_merge_ms = merge_latencies_sorted[len(merge_latencies_sorted) // 2]
    avg_merge_ms    = sum(merge_latencies) / len(merge_latencies)
    print(f"    Số micro-batch đo được   : {len(merge_latencies)}")
    print(f"    MERGE latency (avg)      : {avg_merge_ms:.0f} ms")
    print(f"    MERGE latency (median)   : {median_merge_ms:.0f} ms  ← dùng giá trị này cho luận văn")
    results["merge_latency_ms"] = round(median_merge_ms)
else:
    print("    Không đủ snapshot để tính MERGE latency")
    results["merge_latency_ms"] = None

results["silver_total_records"]   = int(total_records)
results["silver_expired_records"] = int(expired_records)
results["silver_active_records"]  = int(active_records)
results["silver_throughput_rps"]  = round(throughput_rps, 1) if throughput_rps else None
results["silver_pipeline_duration_sec"] = round(duration_sec) if duration_sec else None

METRIC 2: SCD2 MERGE THROUGHPUT


  Total records in Silver  : 2,783,056
  Unique users (SCD keys)  : 1,987,898
  Active records (current) : 1,987,898
  Expired (SCD2 history)   : 795,158 (28.6% of total)

  Silver snapshot history:
    Total snapshots          : 262  (2 per micro-batch = MERGE step1 + step2)
    Total added ops          : 191,603,433
    Total deleted ops        : 188,820,377
    Pipeline duration        : 3623s (60.4 phút)
    Throughput (records/s)   : 768.2  ← dùng giá trị này cho luận văn
    MERGE ops/s (add+del)    : 105,007.0

  MERGE latency per micro-batch (từ Silver snapshot intervals):
    Số micro-batch đo được   : 131
    MERGE latency (avg)      : 2269 ms
    MERGE latency (median)   : 2081 ms  ← dùng giá trị này cho luận văn


---
## Metric 3 — Compaction Impact (Iceberg REWRITE DATA FILES)

Streaming sinh ra nhiều small files — đây là vấn đề phổ biến trong mọi hệ thống streaming.
Compaction gộp các small files thành ít file lớn hơn, cải thiện read performance đáng kể.

In [4]:
print("=" * 55)
print("METRIC 3A: FILE STATS TRƯỚC COMPACTION")
print("=" * 55)

def get_file_stats(table_name):
    """Lấy số lượng và size của data files trong Iceberg table."""
    stats = spark.sql(f"""
        SELECT
            COUNT(*)                 AS file_count,
            SUM(file_size_in_bytes)  AS total_bytes,
            AVG(file_size_in_bytes)  AS avg_bytes,
            MIN(file_size_in_bytes)  AS min_bytes,
            MAX(file_size_in_bytes)  AS max_bytes,
            SUM(record_count)        AS total_records
        FROM {table_name}.files
    """).collect()[0]
    return stats

def measure_query_time(query, label, n_runs=3):
    """Chạy query n lần và lấy median latency."""
    times = []
    for i in range(n_runs):
        t0 = time.time()
        spark.sql(query).collect()
        t1 = time.time()
        times.append((t1 - t0) * 1000)
    times.sort()
    median_ms = times[len(times) // 2]
    print(f"  {label}: {median_ms:.0f} ms (median of {n_runs} runs)")
    return median_ms

BENCH_QUERY_SILVER = """
    SELECT
        is_current,
        COUNT(*) as cnt,
        AVG(review_count) as avg_reviews,
        AVG(fans) as avg_fans
    FROM nessie.silver.yelp_users_scd2
    GROUP BY is_current
"""

BENCH_QUERY_GOLD = """
    SELECT
        state,
        COUNT(*) as total,
        ROUND(AVG(stars), 2) as avg_stars,
        SUM(CAST(attr_outdoor_seating AS INT)) as outdoor_count
    FROM nessie.gold.businesses_flat
    WHERE is_open = 1
    GROUP BY state
    ORDER BY total DESC
"""

silver_before = get_file_stats("nessie.silver.yelp_users_scd2")
gold_before   = get_file_stats("nessie.gold.businesses_flat")

print(f"  [Silver] Files: {silver_before['file_count']:,} | "
      f"Total: {silver_before['total_bytes']/1024/1024:.1f} MB | "
      f"Avg: {silver_before['avg_bytes']/1024:.1f} KB | "
      f"Min: {silver_before['min_bytes']/1024:.1f} KB")

print(f"  [Gold]   Files: {gold_before['file_count']:,} | "
      f"Total: {gold_before['total_bytes']/1024/1024:.1f} MB | "
      f"Avg: {gold_before['avg_bytes']/1024:.1f} KB | "
      f"Min: {gold_before['min_bytes']/1024:.1f} KB")

print()
print("  Query time trước compaction:")
qtime_silver_before = measure_query_time(BENCH_QUERY_SILVER, "Silver aggregate")
qtime_gold_before   = measure_query_time(BENCH_QUERY_GOLD,   "Gold aggregate  ")

results["silver_files_before"]      = int(silver_before["file_count"])
results["gold_files_before"]        = int(gold_before["file_count"])
results["silver_query_before_ms"]   = round(qtime_silver_before)
results["gold_query_before_ms"]     = round(qtime_gold_before)

METRIC 3A: FILE STATS TRƯỚC COMPACTION
  [Silver] Files: 371 | Total: 85.7 MB | Avg: 236.6 KB | Min: 3.9 KB
  [Gold]   Files: 12 | Total: 11.3 MB | Avg: 967.3 KB | Min: 609.8 KB

  Query time trước compaction:


  Silver aggregate: 1677 ms (median of 3 runs)
  Gold aggregate  : 350 ms (median of 3 runs)


In [5]:
print("=" * 55)
print("METRIC 3B: COMPACTION — REWRITE DATA FILES")
print("=" * 55)

# Compaction Silver
print("  [Silver] Đang compact...")
t0 = time.time()
spark.sql("""
    CALL nessie.system.rewrite_data_files(
        table  => 'nessie.silver.yelp_users_scd2',
        options => map(
            'target-file-size-bytes', '134217728',
            'min-input-files', '2'
        )
    )
""").show()
silver_compact_time = time.time() - t0
print(f"  [Silver] Compaction xong: {silver_compact_time:.1f}s")

# Compaction Gold
print()
print("  [Gold] Đang compact...")
t0 = time.time()
spark.sql("""
    CALL nessie.system.rewrite_data_files(
        table  => 'nessie.gold.businesses_flat',
        options => map(
            'target-file-size-bytes', '134217728',
            'min-input-files', '2'
        )
    )
""").show()
gold_compact_time = time.time() - t0
print(f"  [Gold] Compaction xong: {gold_compact_time:.1f}s")

results["silver_compaction_time_sec"] = round(silver_compact_time, 1)
results["gold_compaction_time_sec"]   = round(gold_compact_time, 1)

METRIC 3B: COMPACTION — REWRITE DATA FILES
  [Silver] Đang compact...


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                       371|                     1|             89888806|                      0|
+--------------------------+----------------------+---------------------+-----------------------+

  [Silver] Compaction xong: 12.7s

  [Gold] Đang compact...


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                        12|                     1|             11885650|                      0|
+--------------------------+----------------------+---------------------+-----------------------+

  [Gold] Compaction xong: 3.5s


In [6]:
print("=" * 55)
print("METRIC 3C: FILE STATS SAU COMPACTION")
print("=" * 55)

silver_after = get_file_stats("nessie.silver.yelp_users_scd2")
gold_after   = get_file_stats("nessie.gold.businesses_flat")

print(f"  [Silver] Files: {silver_after['file_count']:,} | "
      f"Total: {silver_after['total_bytes']/1024/1024:.1f} MB | "
      f"Avg: {silver_after['avg_bytes']/1024:.1f} KB")

print(f"  [Gold]   Files: {gold_after['file_count']:,} | "
      f"Total: {gold_after['total_bytes']/1024/1024:.1f} MB | "
      f"Avg: {gold_after['avg_bytes']/1024:.1f} KB")

print()
print("  Query time sau compaction:")
# Warm-up: flush executor cache sau compaction trước khi đo
# (tránh trường hợp query after chậm hơn before do cold cache)
print("  [Warm-up] Đang flush cache...")
spark.sql("SELECT COUNT(*) FROM nessie.silver.yelp_users_scd2").collect()
spark.sql("SELECT COUNT(*) FROM nessie.gold.businesses_flat").collect()
time.sleep(3)
print("  [Warm-up] Done — bắt đầu đo...")
qtime_silver_after = measure_query_time(BENCH_QUERY_SILVER, "Silver aggregate")
qtime_gold_after   = measure_query_time(BENCH_QUERY_GOLD,   "Gold aggregate  ")

silver_file_reduction = (1 - silver_after['file_count'] / silver_before['file_count']) * 100
gold_file_reduction   = (1 - gold_after['file_count']   / gold_before['file_count'])   * 100
silver_query_speedup  = (1 - qtime_silver_after / qtime_silver_before) * 100
gold_query_speedup    = (1 - qtime_gold_after   / qtime_gold_before)   * 100

print()
print("  === IMPROVEMENT ===")
print(f"  Silver: files giảm {silver_file_reduction:.0f}% | query nhanh hơn {silver_query_speedup:.0f}%")
print(f"  Gold  : files giảm {gold_file_reduction:.0f}% | query nhanh hơn {gold_query_speedup:.0f}%")

results["silver_files_after"]        = int(silver_after["file_count"])
results["gold_files_after"]          = int(gold_after["file_count"])
results["silver_query_after_ms"]     = round(qtime_silver_after)
results["gold_query_after_ms"]       = round(qtime_gold_after)
results["silver_file_reduction_pct"] = round(silver_file_reduction, 1)
results["gold_file_reduction_pct"]   = round(gold_file_reduction, 1)
results["silver_query_speedup_pct"]  = round(silver_query_speedup, 1)
results["gold_query_speedup_pct"]    = round(gold_query_speedup, 1)

METRIC 3C: FILE STATS SAU COMPACTION
  [Silver] Files: 1 | Total: 82.9 MB | Avg: 84919.5 KB
  [Gold]   Files: 1 | Total: 10.7 MB | Avg: 10949.1 KB

  Query time sau compaction:
  [Warm-up] Đang flush cache...
  [Warm-up] Done — bắt đầu đo...
  Silver aggregate: 379 ms (median of 3 runs)
  Gold aggregate  : 341 ms (median of 3 runs)

  === IMPROVEMENT ===
  Silver: files giảm 100% | query nhanh hơn 77%
  Gold  : files giảm 92% | query nhanh hơn 2%


---
## Metric 4 — Gold Query Latency (Analytics Queries)

In [7]:
print("=" * 55)
print("METRIC 4: GOLD LAYER QUERY LATENCY")
print("=" * 55)

queries = {
    "Q1_count_scan":
        "SELECT COUNT(*) FROM nessie.gold.businesses_flat",

    "Q2_group_by_state":
        """
        SELECT state, COUNT(*) as cnt, ROUND(AVG(stars),2) as avg_stars
        FROM nessie.gold.businesses_flat
        GROUP BY state ORDER BY cnt DESC
        """,

    "Q3_filter_multi_attr":
        """
        SELECT name, city, stars, review_count
        FROM nessie.gold.businesses_flat
        WHERE attr_outdoor_seating = true
          AND attr_restaurants_delivery = true
          AND is_open = 1
        ORDER BY stars DESC
        LIMIT 20
        """,

    "Q4_category_explode":
        """
        SELECT cat, COUNT(*) as cnt
        FROM nessie.gold.businesses_flat
        LATERAL VIEW explode(categories) tmp AS cat
        GROUP BY cat
        ORDER BY cnt DESC
        LIMIT 20
        """,

    "Q5_scd2_history_join":
        """
        SELECT
            u.user_id,
            COUNT(*) as version_count,
            MIN(u.review_count) as min_reviews,
            MAX(u.review_count) as max_reviews
        FROM nessie.silver.yelp_users_scd2 u
        GROUP BY u.user_id
        HAVING COUNT(*) > 1
        ORDER BY version_count DESC
        LIMIT 10
        """
}

query_results = {}
for name, q in queries.items():
    ms = measure_query_time(q, name, n_runs=3)
    query_results[name] = round(ms)

results["gold_queries_ms"] = query_results

print()
print("  Chú thích:")
print("  Q1: Full scan + count (đo I/O baseline)")
print("  Q2: GROUP BY state (đo aggregation)")
print("  Q3: Multi-column filter (đo predicate pushdown)")
print("  Q4: LATERAL VIEW explode array (đo array processing)")
print("  Q5: SCD2 history query — full scan Silver + GROUP BY + HAVING.")
print("      Latency cao hơn Q1-Q4 là trade-off bình thường của SCD2 pattern:")
print("      mỗi user_id có nhiều version row → không thể predicate pushdown.")
print("      Đây là chi phí của việc lưu lịch sử thay đổi (history preservation).")

METRIC 4: GOLD LAYER QUERY LATENCY
  Q1_count_scan: 94 ms (median of 3 runs)
  Q2_group_by_state: 266 ms (median of 3 runs)
  Q3_filter_multi_attr: 144 ms (median of 3 runs)
  Q4_category_explode: 265 ms (median of 3 runs)


  Q5_scd2_history_join: 2690 ms (median of 3 runs)

  Chú thích:
  Q1: Full scan + count (đo I/O baseline)
  Q2: GROUP BY state (đo aggregation)
  Q3: Multi-column filter (đo predicate pushdown)
  Q4: LATERAL VIEW explode array (đo array processing)
  Q5: SCD2 history query — full scan Silver + GROUP BY + HAVING.
      Latency cao hơn Q1-Q4 là trade-off bình thường của SCD2 pattern:
      mỗi user_id có nhiều version row → không thể predicate pushdown.
      Đây là chi phí của việc lưu lịch sử thay đổi (history preservation).


---
## Tổng hợp kết quả — Bảng cho Luận văn

In [8]:
W = 65

def row(text):
    """In 1 dòng căn trái trong khung, tự động pad."""
    print(f"║  {text:<{W-2}}║")

def sep():
    print("╠" + "═" * W + "╣")

print()
print("╔" + "═" * W + "╗")
row("KẾT QUẢ BENCHMARK — MODERN DATA LAKEHOUSE")
sep()

row("METRIC 1: END-TO-END INGESTION LATENCY (Kafka → Bronze)")
row(f"  Batch interval (median)  : {results.get('ingestion_latency_sec', 'N/A')} giây")
row(f"  Batch interval (avg)     : {results.get('ingestion_latency_avg_sec', 'N/A')} giây")
row(f"  Batch interval (min/max) : {results.get('ingestion_latency_min_sec','N/A')}s / {results.get('ingestion_latency_max_sec','N/A')}s")
row(f"  Trigger config           : 15s")
sep()

row("METRIC 2: SILVER SCD2 THROUGHPUT")
row(f"  Total records Silver     : {results.get('silver_total_records', 'N/A'):,}" if isinstance(results.get('silver_total_records'), int) else f"  Total records Silver     : {results.get('silver_total_records', 'N/A')}")
row(f"  Active (is_current=true) : {results.get('silver_active_records', 'N/A'):,}" if isinstance(results.get('silver_active_records'), int) else f"  Active (is_current=true) : {results.get('silver_active_records', 'N/A')}")
row(f"  Expired (SCD2 history)   : {results.get('silver_expired_records', 'N/A'):,}" if isinstance(results.get('silver_expired_records'), int) else f"  Expired (SCD2 history)   : {results.get('silver_expired_records', 'N/A')}")
row(f"  Pipeline duration        : {results.get('silver_pipeline_duration_sec', 'N/A')}s")
row(f"  Avg throughput           : {results.get('silver_throughput_rps', 'N/A')} records/s")
row(f"  MERGE latency (median)   : {results.get('merge_latency_ms', 'N/A')} ms")
sep()

row("METRIC 3: COMPACTION IMPACT")
row(f"  Silver : {results.get('silver_files_before','?')} files → {results.get('silver_files_after','?')} files  (-{results.get('silver_file_reduction_pct','?')}%)")
row(f"  Silver query : {results.get('silver_query_before_ms','?')}ms → {results.get('silver_query_after_ms','?')}ms  ({results.get('silver_query_speedup_pct','?')}% faster)")
row(f"  Gold   : {results.get('gold_files_before','?')} files → {results.get('gold_files_after','?')} files  (-{results.get('gold_file_reduction_pct','?')}%)")
row(f"  Gold query   : {results.get('gold_query_before_ms','?')}ms → {results.get('gold_query_after_ms','?')}ms  ({results.get('gold_query_speedup_pct','?')}% faster)")
sep()

row("METRIC 4: GOLD QUERY LATENCY (sau compaction)")
if "gold_queries_ms" in results:
    for qname, ms in results["gold_queries_ms"].items():
        row(f"  {qname:<32}: {ms:>6} ms")

print("╚" + "═" * W + "╝")

print()
print("Lưu kết quả vào file benchmark_results.json...")
import json as _json
with open("/home/jovyan/benchmark_results.json", "w") as f:
    _json.dump(results, f, indent=2, default=str)
print("✅ Saved: /home/jovyan/benchmark_results.json")


╔═════════════════════════════════════════════════════════════════╗
║  KẾT QUẢ BENCHMARK — MODERN DATA LAKEHOUSE                      ║
╠═════════════════════════════════════════════════════════════════╣
║  METRIC 1: END-TO-END INGESTION LATENCY (Kafka → Bronze)        ║
║    Batch interval (median)  : 15.0 giây                         ║
║    Batch interval (avg)     : 21.2 giây                         ║
║    Batch interval (min/max) : 13.5s / 749.5s                    ║
║    Trigger config           : 15s                               ║
╠═════════════════════════════════════════════════════════════════╣
║  METRIC 2: SILVER SCD2 THROUGHPUT                               ║
║    Total records Silver     : 2,783,056                         ║
║    Active (is_current=true) : 1,987,898                         ║
║    Expired (SCD2 history)   : 795,158                           ║
║    Pipeline duration        : 3623s                             ║
║    Avg throughput           : 768.2 records/s